# DisasterLens on a Colab GPU

This notebook pulls the repository into the Colab runtime, mounts the official BRIGHT copy in Drive, then runs the real-data M1 audit and M2 eight-tile overfit gate. It never creates a substitute dataset.

In [ ]:
import os
import subprocess
import base64
from getpass import getpass
from pathlib import Path

github_token = os.environ.get("GITHUB_TOKEN")
if not github_token:
    try:
        from google.colab import userdata
        github_token = userdata.get("GITHUB_TOKEN")
    except Exception:
        github_token = None
if not github_token:
    github_token = getpass("GitHub token (input is hidden): ")
github_token = github_token.strip()
if not github_token:
    raise RuntimeError("No GitHub token was supplied.")

REPO_URL = os.environ.get("DISASTERLENS_REPO_URL", "https://github.com/kushc2004/disaster-lens.git")
REPO_DIR = Path("/content/disaster-lens")
git_env = os.environ.copy()
basic_auth = base64.b64encode(f"x-access-token:{github_token}".encode()).decode()
git_env.update({"GIT_CONFIG_COUNT": "1", "GIT_CONFIG_KEY_0": "http.extraHeader", "GIT_CONFIG_VALUE_0": f"Authorization: Basic {basic_auth}"})

if REPO_DIR.exists():
    if (REPO_DIR / ".git").exists():
        command = ["git", "-C", str(REPO_DIR), "pull"]
    else:
        import shutil
        shutil.rmtree(REPO_DIR)
        command = ["git", "clone", REPO_URL, str(REPO_DIR)]
else:
    command = ["git", "clone", REPO_URL, str(REPO_DIR)]
result = subprocess.run(command, env=git_env, capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError(f"GitHub command failed ({result.returncode}):\n{result.stderr.strip()}")
print(result.stdout, end="")
%cd /content/disaster-lens

In [ ]:
%pip install -e .

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
assert torch.cuda.is_available(), "Select a Colab GPU kernel before running this notebook."

In [ ]:
# M1: inspect only the official BRIGHT data already extracted in Drive.
from google.colab import drive
from pathlib import Path
import os

drive.mount('/content/drive')
BRIGHT_ROOT = Path('/content/drive/MyDrive/disaster-lens/data/raw/bright')
required = [BRIGHT_ROOT / name for name in ('pre-event', 'post-event', 'target')]
if not all(path.is_dir() for path in required):
    raise RuntimeError(f'Expected extracted official BRIGHT directories at {BRIGHT_ROOT}; found {[str(p) for p in required if not p.is_dir()]}')
os.environ['DISASTERLENS_BRIGHT_ROOT'] = str(BRIGHT_ROOT)
!python scripts/inspect_bright.py data=bright
!python scripts/build_manifest.py data=bright
!python scripts/verify_bright_loader.py data=bright

In [ ]:
# Select one real event from outputs/reports/bright_data_audit.md, then run the M2 gate.
# Do not use a made-up event ID. The value below is deliberately required.
TEST_EVENT = ''
if not TEST_EVENT:
    raise ValueError('Set TEST_EVENT to an event ID printed by the real M1 audit, then re-run this cell.')

!python scripts/make_splits.py data=bright split.test_events=[$TEST_EVENT]
!python scripts/train.py split_path=data/manifests/splits/event_holdout.json overfit_tiles=8 trainer.epochs=100 trainer.crop_size=512
!python scripts/evaluate.py checkpoint=outputs/checkpoints/early_fusion_unet/best.pt split_path=data/manifests/splits/event_holdout.json partition=test